# Notebook 12 — ViT-Based Semantic Segmentation

## Learning Objectives
- Adapt a Vision Transformer for dense pixel-wise prediction (segmentation).
- Understand how patch tokens are decoded back to full-resolution masks.
- Compare ViT-Segmenter vs U-Net on the same synthetic shapes dataset.
- Analyse the trade-off between global attention (ViT) and skip connections (U-Net).
- Identify when global context is helpful vs when local precision matters.

## Big Picture
Classification ViT reads one label from the CLS token. For segmentation, we need a label at
every pixel. The solution: discard the CLS token and reshape the patch tokens back into a 2D
feature map, then upsample with a convolutional decoder.

## Dataset
Same **SyntheticShapes** segmentation dataset as Notebook 11 (128×128, 4 classes).

## Architecture
```
Input [B, 3, 128, 128]
  └─ PatchEmbedding (patch_size=16): 8×8=64 patches
     each patch: 16×16×3=768 pixels → Linear(768→64)  → [B, 64, 64]
  └─ + SinusoidalPositionalEncoding                   → [B, 64, 64]
  └─ TransformerEncoder (2 blocks, 4 heads)           → [B, 64, 64]
  └─ Reshape tokens: [B, 64, 8, 8]                    (d_model channels, 8×8 spatial)
  └─ ConvTranspose decoder (upsample by 16×):
       ConvTranspose2d(64→16, k=2, s=2)  → [B, 16, 16, 16]
       BN + ReLU
       ConvTranspose2d(16→16, k=2, s=2)  → [B, 16, 32, 32]
       BN + ReLU
       ConvTranspose2d(16→16, k=2, s=2)  → [B, 16, 64, 64]
       BN + ReLU
       ConvTranspose2d(16→4,  k=2, s=2)  → [B,  4, 128, 128]  (logits)
```

## Theory
**ViT-based segmentation** replaces the U-Net encoder with a Transformer that attends globally
to all patches. This is beneficial when:
- Global context is needed (e.g., understanding scene layout).
- Shapes have long-range dependencies (a piece of sky above water).

**Without skip connections**, the ViT decoder upsamples from a coarse feature grid. This can
be less precise at boundaries than U-Net's skip connections, which directly provide high-res
encoder features to the decoder.

**Patch size trade-off**:
- Smaller patches → more tokens → finer spatial resolution → better detail, but $O(N^2)$ attention cost.
- Larger patches → fewer tokens → coarser spatial resolution → faster, but less precise.

Here we use `patch_size=16` on 128×128 images → 8×8=64 tokens.

## Implementation from Scratch
### 1. Imports and Setup

In [1]:
import sys
sys.path.insert(0, '..')  # add repo root when running from notebooks/

import json
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_metrics_json, save_markdown_report, update_runs_summary, save_table_csv
from src.utils.paths import RUNS_SEGMENTATION_DIR
from src.data.synthetic_shapes import load_shapes_data
from src.models.vit_segmenter import ViTSegmenter
from src.metrics.segmentation import pixel_accuracy, mean_iou, dice_score
from src.visualization.segmentation import overlay_segmentation_mask, plot_segmentation_examples

seed_everything(42)
device = get_best_device()
run_dir = prepare_run_dir(RUNS_SEGMENTATION_DIR, clean=False)
print(f'Device  : {device}')
print(f'Run dir : {run_dir}')

[seed] Random seed set to 42.
Device  : cuda
Run dir : /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation


## Dataset
### 2. Load Synthetic Shapes (same as Notebook 11)

In [2]:
IMAGE_SIZE  = 128
PATCH_SIZE  = 16   # 128/16 = 8 patches per side → 64 tokens
NUM_CLASSES = 4
D_MODEL     = 64
NUM_HEADS   = 8
NUM_LAYERS  = 2
DIM_FF      = 128
BATCH_SIZE  = 16
NUM_EPOCHS  = 10
LR          = 6e-4
CLASS_NAMES = ['background', 'circle', 'square', 'triangle']

train_loader, val_loader = load_shapes_data(
    n_train=800,
    n_val=200,
    image_size=IMAGE_SIZE,
    mode='segmentation',
    batch_size=BATCH_SIZE,
    num_workers=0,
    seed=42,
)

imgs, masks = next(iter(train_loader))
print(f'Image batch shape : {imgs.shape}   (B, 3, H, W)')
print(f'Mask  batch shape : {masks.shape}  (B, H, W)')
print(f'Num patches       : {(IMAGE_SIZE//PATCH_SIZE)**2}  ({IMAGE_SIZE//PATCH_SIZE}×{IMAGE_SIZE//PATCH_SIZE} grid)')

Image batch shape : torch.Size([16, 3, 128, 128])   (B, 3, H, W)
Mask  batch shape : torch.Size([16, 128, 128])  (B, H, W)
Num patches       : 64  (8×8 grid)


### 3. Build ViTSegmenter

In [3]:
model = ViTSegmenter(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=3,
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FF,
    dropout=0.1,
).to(device)

n_params = model.count_parameters()
print(f'Trainable parameters: {n_params:,}')

# Shape trace
x_sample = imgs[:2].to(device)          # [2, 3, 128, 128]
with torch.no_grad():
    logits = model(x_sample)             # [2, 4, 128, 128]

print(f'Input  shape : {x_sample.shape}')
print(f'Output shape : {logits.shape}  (same spatial size as input — check!)')
print(f'Num patches  : {model.num_patches}  ({model.num_patches_per_side}×{model.num_patches_per_side})')

Trainable parameters: 122,324
Input  shape : torch.Size([2, 3, 128, 128])
Output shape : torch.Size([2, 4, 128, 128])  (same spatial size as input — check!)
Num patches  : 64  (8×8)


## Sanity Checks

In [4]:
assert logits.shape == (2, NUM_CLASSES, IMAGE_SIZE, IMAGE_SIZE), f'Shape mismatch: {logits.shape}'
print('Output shape check passed.')

assert not torch.isnan(logits).any()
print('No NaN in output.')

preds = logits.argmax(dim=1)
masks_dev = masks[:2].to(device)
pa = pixel_accuracy(preds, masks_dev)
print(f'Random-init pixel accuracy : {pa:.3f}  (expected ~0.25 for 4 classes)')

loss_val = F.cross_entropy(logits, masks_dev)
print(f'Initial loss : {loss_val.item():.4f}')
print('All sanity checks passed!')

Output shape check passed.
No NaN in output.
Random-init pixel accuracy : 0.103  (expected ~0.25 for 4 classes)
Initial loss : 1.6634
All sanity checks passed!


## Training

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'pixel_accuracy': [], 'mean_iou': [], 'dice': []}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, n_batches = 0.0, 0
    for batch_imgs, batch_masks in train_loader:
        batch_imgs  = batch_imgs.to(device)   # [B, 3, 128, 128]
        batch_masks = batch_masks.to(device)  # [B, 128, 128]
        optimizer.zero_grad()
        logits = model(batch_imgs)             # [B, 4, 128, 128]
        loss = F.cross_entropy(logits, batch_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()
        n_batches += 1
    history['train_loss'].append(t_loss / n_batches)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_pa, v_miou, v_dice, v_n = 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for batch_imgs, batch_masks in val_loader:
            batch_imgs  = batch_imgs.to(device)
            batch_masks = batch_masks.to(device)
            logits = model(batch_imgs)
            preds  = logits.argmax(dim=1)
            v_loss += F.cross_entropy(logits, batch_masks).item()
            v_pa   += pixel_accuracy(preds, batch_masks)
            v_miou += mean_iou(preds, batch_masks, num_classes=NUM_CLASSES)
            v_dice += dice_score(preds, batch_masks, num_classes=NUM_CLASSES)
            v_n += 1
    history['val_loss'].append(v_loss / v_n)
    history['pixel_accuracy'].append(v_pa / v_n)
    history['mean_iou'].append(v_miou / v_n)
    history['dice'].append(v_dice / v_n)
    scheduler.step()

    print(f'Epoch {epoch}/{NUM_EPOCHS}  '
          f'train={history["train_loss"][-1]:.4f}  '
          f'val={history["val_loss"][-1]:.4f}  '
          f'pxacc={history["pixel_accuracy"][-1]:.4f}  '
          f'mIoU={history["mean_iou"][-1]:.4f}  '
          f'dice={history["dice"][-1]:.4f}')

print('Training complete!')

Epoch 1/10  train=1.4679  val=1.3011  pxacc=0.3118  mIoU=0.1217  dice=0.2022
Epoch 2/10  train=1.1241  val=0.9369  pxacc=0.7954  mIoU=0.3805  dice=0.4906
Epoch 3/10  train=0.8114  val=0.6597  pxacc=0.9404  mIoU=0.5785  dice=0.6587
Epoch 4/10  train=0.5922  val=0.4987  pxacc=0.9466  mIoU=0.5959  dice=0.6598
Epoch 5/10  train=0.4582  val=0.3899  pxacc=0.9506  mIoU=0.6093  dice=0.6683
Epoch 6/10  train=0.3795  val=0.3374  pxacc=0.9525  mIoU=0.6141  dice=0.6708
Epoch 7/10  train=0.3358  val=0.2992  pxacc=0.9536  mIoU=0.6187  dice=0.6740
Epoch 8/10  train=0.3103  val=0.2873  pxacc=0.9539  mIoU=0.6174  dice=0.6719
Epoch 9/10  train=0.2972  val=0.2756  pxacc=0.9543  mIoU=0.6197  dice=0.6731
Epoch 10/10  train=0.2930  val=0.2755  pxacc=0.9542  mIoU=0.6198  dice=0.6725
Training complete!


## Evaluation

In [6]:
# Final evaluation
model.eval()
total_pa, total_miou, total_dice, n = 0.0, 0.0, 0.0, 0
with torch.no_grad():
    for batch_imgs, batch_masks in val_loader:
        preds = model(batch_imgs.to(device)).argmax(dim=1)
        total_pa   += pixel_accuracy(preds, batch_masks.to(device))
        total_miou += mean_iou(preds, batch_masks.to(device), num_classes=NUM_CLASSES)
        total_dice += dice_score(preds, batch_masks.to(device), num_classes=NUM_CLASSES)
        n += 1

final_pa   = total_pa   / n
final_miou = total_miou / n
final_dice = total_dice / n

print(f'ViT-Segmenter Pixel Accuracy : {final_pa:.4f}')
print(f'ViT-Segmenter Mean IoU       : {final_miou:.4f}')
print(f'ViT-Segmenter Dice Score     : {final_dice:.4f}')

vit_metrics = {
    'pixel_accuracy': float(final_pa),
    'mean_iou': float(final_miou),
    'dice_score': float(final_dice),
    'final_train_loss': float(history['train_loss'][-1]),
    'final_val_loss': float(history['val_loss'][-1]),
    'num_params': n_params,
    'num_epochs': NUM_EPOCHS,
}
save_metrics_json(vit_metrics, run_dir / 'vit_segmenter_metrics.json')
print(f'Metrics saved.')

ViT-Segmenter Pixel Accuracy : 0.9542
ViT-Segmenter Mean IoU       : 0.6198
ViT-Segmenter Dice Score     : 0.6725
Metrics saved.


In [7]:
# Compare with U-Net results
unet_metrics_path = run_dir / 'unet_metrics.json'
comparison_rows = []

if unet_metrics_path.exists():
    with open(unet_metrics_path) as f:
        unet_m = json.load(f)
    print('=== U-Net vs ViT-Segmenter Comparison ===')
    print(f'{"Model":<20} {"Px Acc":>8} {"mIoU":>8} {"Dice":>8} {"Params":>10}')
    print('-' * 60)
    print(f'{"U-Net":<20} {unet_m["pixel_accuracy"]:>8.4f} {unet_m["mean_iou"]:>8.4f} '
          f'{unet_m["dice_score"]:>8.4f} {unet_m["num_params"]:>10,}')
    print(f'{"ViT-Segmenter":<20} {final_pa:>8.4f} {final_miou:>8.4f} '
          f'{final_dice:>8.4f} {n_params:>10,}')

    comparison_rows = [
        {'model': 'U-Net', 'pixel_accuracy': unet_m['pixel_accuracy'],
         'mean_iou': unet_m['mean_iou'], 'dice_score': unet_m['dice_score'],
         'num_params': unet_m['num_params'], 'epochs': unet_m['num_epochs']},
        {'model': 'ViT-Segmenter', 'pixel_accuracy': final_pa,
         'mean_iou': final_miou, 'dice_score': final_dice,
         'num_params': n_params, 'epochs': NUM_EPOCHS},
    ]
else:
    print('U-Net metrics not found — run Notebook 11 first.')
    comparison_rows = [
        {'model': 'ViT-Segmenter', 'pixel_accuracy': final_pa,
         'mean_iou': final_miou, 'dice_score': final_dice,
         'num_params': n_params, 'epochs': NUM_EPOCHS},
    ]

# Save comparison CSV
save_table_csv(comparison_rows, run_dir / 'vit_vs_unet_comparison.csv')
print(f'Comparison CSV saved to: {run_dir}/vit_vs_unet_comparison.csv')

=== U-Net vs ViT-Segmenter Comparison ===
Model                  Px Acc     mIoU     Dice     Params
------------------------------------------------------------
U-Net                  1.0000   0.9998   0.9999  1,948,388
ViT-Segmenter          0.9542   0.6198   0.6725    122,324
Comparison CSV saved to: /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_vs_unet_comparison.csv


## Visualization

In [8]:
# Training curves
from src.visualization.plots import plot_training_curves
plot_training_curves(
    {'train_loss': history['train_loss'], 'val_loss': history['val_loss'],
     'mean_iou': history['mean_iou'], 'dice': history['dice']},
    save_path=run_dir / 'vit_segmenter_training_curve.png',
    title='ViT-Segmenter Training Curves (Synthetic Shapes)',
)
print('Training curve saved.')

Training curve saved.


In [9]:
# Segmentation examples
model.eval()
vis_imgs, vis_masks = next(iter(val_loader))
with torch.no_grad():
    vis_preds = model(vis_imgs.to(device)).argmax(dim=1).cpu()

overlay_segmentation_mask(
    image=vis_imgs[0].numpy(),
    mask=vis_preds[0].numpy(),
    alpha=0.5,
    save_path=run_dir / 'vit_segmenter_overlay.png',
    title='ViT-Segmenter Predicted Segmentation Overlay',
    class_names=['background', 'circle', 'square', 'triangle'],
)

plot_segmentation_examples(
    images=vis_imgs.numpy(),
    gt_masks=vis_masks.numpy(),
    pred_masks=vis_preds.numpy(),
    save_path=run_dir / 'vit_segmenter_examples.png',
    title='ViT-Segmenter: Image | GT Mask | Predicted Mask',
    n_examples=4,
)
print('Visualisations saved.')

Visualisations saved.


In [10]:
# Side-by-side comparison bar chart
if len(comparison_rows) == 2:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metric_names = ['pixel_accuracy', 'mean_iou', 'dice_score']
    metric_labels = ['Pixel Accuracy', 'Mean IoU', 'Dice Score']
    model_labels = [r['model'] for r in comparison_rows]
    colors = ['steelblue', 'tomato']

    for ax, mname, mlabel in zip(axes, metric_names, metric_labels):
        vals = [r[mname] for r in comparison_rows]
        bars = ax.bar(model_labels, vals, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=10)
        ax.set_ylim(0, 1.1)
        ax.set_title(mlabel)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('U-Net vs ViT-Segmenter — Synthetic Shapes')
    plt.tight_layout()
    fig.savefig(run_dir / 'unet_vs_vit_metrics_comparison.png', dpi=120)
    plt.close(fig)
    print('Comparison chart saved.')

Comparison chart saved.


## Failure Cases

**Why ViT-Segmenter may underperform U-Net on this dataset:**

1. **No skip connections**: U-Net passes high-resolution encoder features directly to the decoder.
   ViT decodes from only the 8×8 patch token grid. Boundary precision is inherently limited by
   the patch resolution.

2. **Patch boundary artifacts**: Predictions often show grid-aligned artifacts at patch boundaries
   (every 16 pixels), because the model processes each patch independently before attention.

3. **Dataset size**: ViT requires more data to learn spatial inductive biases from scratch.
   With only 800 training images, U-Net's CNN encoder is more data-efficient.

4. **Upsampling by 16×**: Going from 8×8 to 128×128 requires 4 ConvTranspose operations,
   each doubling resolution. Small errors at the patch level are amplified.

**When ViT-Segmenter wins:**
- Large datasets with complex textures and long-range context.
- Pre-trained ViT backbone (e.g., Swin Transformer, ViT-Large).
- Tasks requiring global context (sky/water distinctions, whole-scene understanding).

## Exercises

1. **Smaller patch size**: Change `patch_size` to 8 (16×16=256 tokens). How does mean IoU
   change? What is the training time cost?

2. **Hybrid model**: Add skip connections from patch embedding to the ConvTranspose decoder
   (similar to U-Net). This is the key idea behind SETR-PUP. Does it improve boundaries?

3. **Attention map visualisation**: Extract attention weights from the last encoder block.
   For one patch, display which other patches it attends to most. Do shape pixels attend
   to each other more than to background?

4. **Ablation — number of layers**: Train with `num_layers=1` vs `num_layers=4`. Plot mean
   IoU vs num_layers. Is there diminishing return?

5. **Per-class boundary precision**: For each class, compute the IoU only on pixels within
   3 pixels of a shape boundary. Compare U-Net vs ViT boundary IoU.

## Key Takeaways

- **ViT for segmentation**: patch tokens are reshaped into a 2D feature map and upsampled by
  a convolutional decoder — no CLS token needed.
- **Global attention** captures long-range context that CNNs can only achieve in deeper layers.
- **Without skip connections**, ViT-Segmenter is less precise at boundaries than U-Net.
- **Patch size** controls the resolution-efficiency tradeoff: smaller patches = more detail
  but $O(N^2)$ attention cost.
- Modern segmentation models (Swin, SegFormer, Mask2Former) use hierarchical ViTs with
  multi-scale features — combining ViT's global context with CNN-style local precision.
- The comparison CSV (`vit_vs_unet_comparison.csv`) is used in the final summary (Notebook 16).

## Saved Outputs

In [11]:
save_markdown_report(
    title='ViT-Segmenter — Semantic Segmentation',
    summary=(
        f'ViTSegmenter (image_size=128, patch_size=16, d_model=64) trained for {NUM_EPOCHS} epochs. '
        f'Pixel acc: {final_pa:.4f}, mean IoU: {final_miou:.4f}, Dice: {final_dice:.4f}.'
    ),
    metrics=vit_metrics,
    figures=['vit_segmenter_training_curve.png', 'vit_segmenter_overlay.png',
             'vit_segmenter_examples.png'],
    tables=['vit_vs_unet_comparison.csv'],
    output_path=run_dir / 'vit_segmenter_report.md',
    device=str(device),
    hyperparams={
        'patch_size': PATCH_SIZE, 'd_model': D_MODEL, 'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS, 'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS, 'lr': LR,
    },
    architecture='PatchEmb(16→64) + SinPE + TransformerEncoder×2 → reshape(8×8) → ConvTranspose decoder(×16)',
    loss_fn='CrossEntropyLoss (pixel-wise)',
)

update_runs_summary(
    session_name='vit_segmentation',
    report_path=run_dir / 'vit_segmenter_report.md',
    metrics=vit_metrics,
    figures=['vit_segmenter_training_curve.png', 'vit_segmenter_examples.png'],
)

print('All outputs saved:')
print(f'  {run_dir}/vit_segmenter_metrics.json')
print(f'  {run_dir}/vit_segmenter_training_curve.png')
print(f'  {run_dir}/vit_segmenter_overlay.png')
print(f'  {run_dir}/vit_vs_unet_comparison.csv')
print(f'  {run_dir}/vit_segmenter_report.md')

All outputs saved:
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_metrics.json
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_training_curve.png
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_overlay.png
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_vs_unet_comparison.csv
  /home/arash/PycharmProjects/Transformer/transformers-from-scratch/runs/segmentation/vit_segmenter_report.md
